# Introduction: Analysis of domestic hot water 

In this notebook we seperate the total gas consumption into gas usage for hot water and gas usage for heating. 

Our approach is as follows: 
- Look at consumption during summer period (May 1 - August 1) and ensure we only take days the heatpump was off (< 1kW/d consumption)
- Compute the mean consumption during this period. This is the average daily usage for non-heating, subtract this from all days to be left with heating gas usage.
- For the households that have no valid days during the summer period, take the population (all participants) average

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import datetime, timedelta

# load the combined filtered/imputed dataset - step 3

path_base = "cache/step3/"
df_dataset = pd.read_parquet(f"{path_base}df_filtered_dataset.parquet")

In [2]:
print(df_dataset.dtypes)
df_dataset

participant_id                     object
time                       datetime64[ns]
gas_consumption                   float64
elek_consumption                  float64
supply_temp                       float64
return_temp                       float64
flowrate                          float64
energy_out                        float64
is_first_value                     object
is_incomplete_yesterday            object
is_within_threshold                object
outside_temp                      float64
is_outside_temp_imputed            object
inside_temp                       float64
delta_temp                        float64
energy_gas                        float64
energy_in                         float64
dtype: object


,participant_id,time,gas_consumption,elek_consumption,supply_temp,return_temp,flowrate,energy_out,is_first_value,is_incomplete_yesterday,is_within_threshold,outside_temp,is_outside_temp_imputed,inside_temp,delta_temp,energy_gas,energy_in
0,-B51sQOp,2022-06-14,NaN,NaN,NaN,NaN,0.000000,NaN,None,None,None,NaN,None,25.500000,NaN,NaN,NaN
1,-B51sQOp,2022-11-08,NaN,NaN,29.170000,23.640000,323.411765,NaN,None,None,None,13.700000,False,19.582000,5.882000,NaN,NaN
2,-B51sQOp,2022-11-09,NaN,NaN,29.757273,24.083182,347.645833,NaN,None,None,None,12.954167,False,19.853125,6.898958,NaN,NaN
3,-B51sQOp,2022-11-10,1.036,NaN,29.668158,24.814211,436.173913,38.78,False,False,True,10.854167,False,20.280220,9.426053,10.12172,NaN
4,-B51sQOp,2022-11-11,0.181,NaN,29.745000,25.128636,467.770833,41.81,False,False,True,9.520833,False,20.325000,10.804167,1.76837,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92594,zskLB4tU,2024-05-19,0.231,0.226,NaN,NaN,0.000000,0.00,False,False,True,NaN,None,20.629167,NaN,2.25687,2.48287
92595,zskLB4tU,2024-05-20,0.116,0.226,NaN,NaN,0.000000,0.00,False,False,True,NaN,None,20.244048,NaN,1.13332,1.35932
92596,zskLB4tU,2024-05-21,0.120,0.225,21.950000,19.760000,13.531250,0.00,False,False,True,NaN,None,20.077049,NaN,1.17240,1.39740
92597,zskLB4tU,2024-05-22,0.122,0.228,NaN,NaN,0.000000,0.00,False,False,True,NaN,None,19.990909,NaN,1.19194,1.41994


## Approach for seperating hot water 
In the plots of notebook `data cleaning`, we see that gas consumption stabalizes between may and july, we can assume this period from the start of may (month 5) to the end of july (month 7) is when you only use gas for non heating purposes. This period is taken on the conservative side so we are more certain that we only look at the period with no gas usage for heating. <br>
This can be further refined by filtering out days when the heatpump is idle. This is done using a 1 kWh threshold. <br>
Finally, with this criteria for selecting days. Take the average gas consumption of all these days for the estimated daily average gas usage 
This will be our method for seperating hot water usage. 

Some remarks about this approach: 
- In reality, during winter the hot water gas usage will be higher than in the summer, because the water will be colder by default. 
- We did not take into account the flowrate as not all heatpumps stop their flow during the summer period. (Hence we only use electricity usage to determine if the pump is off)

In [3]:
def plot_dhw(participant_id: str, df_participant: pd.DataFrame, df_summer: pd.DataFrame, column: str, ylabel: str, show_flowrate: bool, gas_mean: int): 
    """Method to plot data with an added line for the gas consumption estimate (non heating) 
    Input:
        - df_participant: all gasconsumption points to plot for this participant
        - df_summer: data only for summer period and heatpump off days
        - column: name of gas consumption column
        - ylabel: y-axis label
        - show_flowrate: make dual axis plot with flowrate 
        - gas_mean: the calculated gas_mean
        
    Returns: 
        None
    """
    plt.figure(figsize=(20,4.5))

    df = df_participant
    
    # Add gas estimate to plot
    plt.plot(df['time'], df['gas_consumption'], color='C0', alpha=1) 
    plt.axhline(y = gas_mean, color = 'r', linestyle = '-', label='Geschat verbruik (niet-verwarmen)') 

    # Start of code block: simplify start - end dates of sampled summer data and add them to plot
    # Instead of one row per day, collapse two 1 row per connected time period (no day gaps),
    if len(df_summer) > 0: 
        # only do this when we actually imputed it from the summer period (instead of the global mean) 
        group_column = 'time'
        df_summer = (
            df_summer.assign(
                group_seq = lambda df: (
                    df_summer[group_column].ne(df[group_column].shift() + timedelta(days=1)).cumsum() 
                )
            )
        ) # For a given day, see if the value of 'time' is the previous date + 1 day, if not: increment group sequence (basically identify gaps)
        df_agg = (
            df_summer.reset_index().groupby("group_seq").agg(
                min = ("time", "min"), max = ("time", "max")
            ).sort_values(["min", "max"])
        ).reset_index() # now use that group_seq(uence) value to group our periods 

        # Add the sampled periods to our plot 
        for row in df_agg.itertuples():
            plt.axvspan(row.min, row.max, color='grey', alpha=0.5, label='Zomerdagen zonder gasverwarming') #Summer days & no heating
        # End of code block 

    # Deduplicate legend labels
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys())

    ax1 = plt.gca()
    ax1.set_ylabel(ylabel) 

    # Use colorblind friendly C10 colors 
    plt.plot(df['time'], df[column], color='C0', alpha=1, label='gas consumption') 

    # Optional: add flowrate to plot. 
    if show_flowrate: 
        ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis

        ax2.set_ylabel('Flow rate') #  we already handled the x-label with ax1
        ax2.plot(df['time'], df['flowrate'], color='C1', alpha=1, label='gas consumption') 
        #ax2.tick_params(axis='y')

    #plt.fill_between(df_quantile['time'], df_quantile['Q1'], df_quantile['Q3'], color='C0', alpha=.25, label='25-75%')
    plt.grid(alpha=.5)
    plt.title(f'Geschat gasverbruik voor deelnemer {participant_id}: {round(gas_mean, 2)} $m^3$')

    # output to general folder
    output_path_png = f'plots/png/schattingen_domestic_hot_water/{participant_id}.png'
    plt.savefig(output_path_png, dpi=150, transparent=False, facecolor='w', bbox_inches='tight')

    # output per participant
    Path(f"plots/png/grafieken_per_deelnemer/{participant_id}").mkdir(parents=True, exist_ok=True)  
    output_path_png = f'plots/png/grafieken_per_deelnemer/{participant_id}/schatting_domestic_hot_water_{participant_id}.png'
    plt.savefig(output_path_png, dpi=150, transparent=False, facecolor='w', bbox_inches='tight')
    plt.close() 
    
def estimate_dhw(df_input: pd.DataFrame, column: str, ylabel: str, show_flowrate: bool): 
    """Method to estimate and plot hot water usage
        
    Returns: 
        None
    """
    Path("plots/png/schattingen_domestic_hot_water").mkdir(parents=True, exist_ok=True)  
    df_grouped = df_input.groupby('participant_id')
        
    ids_with_error = []  
    hot_water_estimates = [] 
    
    for participant_id, df in df_grouped:
        # Filter on summer days with gas consumption
        df_summer = df[(df['time'].dt.month > 5) & (df['time'].dt.month < 8) & (df['gas_consumption'].notna())]
        # Filter on when heatpump is off or we did not no if it was on or not 
        df_summer = df_summer[(df_summer['elek_consumption'] < 1) | (df_summer['elek_consumption'].isna())]
         
        # Calculate and plot gas usage during the summer. This is our estimate for hot water usage. 
        avg_gas_usage = df_summer[['gas_consumption']].mean().values[0]
        #print(avg_gas_usage)
        
        if np.isnan(avg_gas_usage): 
            # skip if we could not determine average 
            ids_with_error.append({'participant_id': participant_id})
            print(f'[Warning] Could not estimate gas usage for id: {participant_id}, using global mean instead') 
        else: 
            hot_water_estimates.append({"participant_id": participant_id, "gas_consumption_hot_water_per_day": avg_gas_usage, 'is_imputed_from_mean_hot_water': False})
            plot_dhw(participant_id, df, df_summer, column, ylabel, show_flowrate, avg_gas_usage) 
        
    
    # for ids with error, use global population average
    global_mean = pd.DataFrame(hot_water_estimates)[['gas_consumption_hot_water_per_day']].mean().values[0]
    df_filtered = pd.DataFrame(ids_with_error).merge(df_input,how='inner', on='participant_id') 
    
    for participant_id, df in df_filtered.groupby('participant_id'): 
        hot_water_estimates.append({"participant_id": participant_id, "gas_consumption_hot_water_per_day": global_mean, 'is_imputed_from_mean_hot_water': True})
        plot_dhw(participant_id, df, pd.DataFrame(), column, ylabel, show_flowrate, global_mean) 

    return pd.DataFrame(hot_water_estimates)

In [4]:
df_estimates = estimate_dhw(df_dataset, column = 'gas_consumption', ylabel='Gasverbruik $m^3/d$', show_flowrate=False)
df_estimates

[Warning] Could not estimate gas usage for id: 0rE1EXuT, using global mean instead
[Warning] Could not estimate gas usage for id: 5QvgGmut, using global mean instead
[Warning] Could not estimate gas usage for id: 66lZ2dKl, using global mean instead
[Warning] Could not estimate gas usage for id: 7JgOwuUh, using global mean instead
[Warning] Could not estimate gas usage for id: 9H8W2b5Q, using global mean instead
[Warning] Could not estimate gas usage for id: IF9c6c-H, using global mean instead
[Warning] Could not estimate gas usage for id: JqpjoDDU, using global mean instead
[Warning] Could not estimate gas usage for id: Jr9PJjgK, using global mean instead
[Warning] Could not estimate gas usage for id: Nkw-o7MK, using global mean instead
[Warning] Could not estimate gas usage for id: TkMyAOp1, using global mean instead
[Warning] Could not estimate gas usage for id: gmtO-ZIa, using global mean instead
[Warning] Could not estimate gas usage for id: qBr4JQ_w, using global mean instead
[War

,participant_id,gas_consumption_hot_water_per_day,is_imputed_from_mean_hot_water
0,-B51sQOp,0.680426,False
1,-ORiFQC6,0.280968,False
2,-uFUg8jU,1.764368,False
3,0RQyG6WX,0.133377,False
4,0XvCtU3v,0.443172,False
...,...,...,...
169,qBr4JQ_w,0.476214,True
170,uQLTOCT9,0.476214,True
171,ueGJf9d5,0.476214,True
172,zK4WxULP,0.476214,True


In [5]:
population_mean = df_estimates[['gas_consumption_hot_water_per_day']].dropna().mean().values[0]
print(f"On average, a house holds consumes {population_mean} m^3 gas for hot water per day")
nr_of_imputed = len(df_estimates[df_estimates['is_imputed_from_mean_hot_water']])
print(f"Could not estimate gas usage hot water for {nr_of_imputed} housesholds, using global mean instead")

On average, a house holds consumes 0.47621360663370255 m^3 gas for hot water per day
Could not estimate gas usage hot water for 16 housesholds, using global mean instead


In [6]:
df3 = df_dataset.merge(df_estimates, how='left', on='participant_id')

In [7]:
# recompute energy_gas
df3['gas_consumption_heating'] = (df3['gas_consumption'] - df3['gas_consumption_hot_water_per_day']).clip(lower=0)
df3['energy_gas'] = df3['gas_consumption_heating'] * 9.77
df3['energy_in'] = df3['elek_consumption'] + df3['energy_gas']
df3

,participant_id,time,gas_consumption,elek_consumption,supply_temp,return_temp,flowrate,energy_out,is_first_value,is_incomplete_yesterday,is_within_threshold,outside_temp,is_outside_temp_imputed,inside_temp,delta_temp,energy_gas,energy_in,gas_consumption_hot_water_per_day,is_imputed_from_mean_hot_water,gas_consumption_heating
0,-B51sQOp,2022-06-14,NaN,NaN,NaN,NaN,0.000000,NaN,None,None,None,NaN,None,25.500000,NaN,NaN,NaN,0.680426,False,NaN
1,-B51sQOp,2022-11-08,NaN,NaN,29.170000,23.640000,323.411765,NaN,None,None,None,13.700000,False,19.582000,5.882000,NaN,NaN,0.680426,False,NaN
2,-B51sQOp,2022-11-09,NaN,NaN,29.757273,24.083182,347.645833,NaN,None,None,None,12.954167,False,19.853125,6.898958,NaN,NaN,0.680426,False,NaN
3,-B51sQOp,2022-11-10,1.036,NaN,29.668158,24.814211,436.173913,38.78,False,False,True,10.854167,False,20.280220,9.426053,3.473956,NaN,0.680426,False,0.355574
4,-B51sQOp,2022-11-11,0.181,NaN,29.745000,25.128636,467.770833,41.81,False,False,True,9.520833,False,20.325000,10.804167,0.000000,NaN,0.680426,False,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92594,zskLB4tU,2024-05-19,0.231,0.226,NaN,NaN,0.000000,0.00,False,False,True,NaN,None,20.629167,NaN,0.000000,0.226,0.476214,True,0.000000
92595,zskLB4tU,2024-05-20,0.116,0.226,NaN,NaN,0.000000,0.00,False,False,True,NaN,None,20.244048,NaN,0.000000,0.226,0.476214,True,0.000000
92596,zskLB4tU,2024-05-21,0.120,0.225,21.950000,19.760000,13.531250,0.00,False,False,True,NaN,None,20.077049,NaN,0.000000,0.225,0.476214,True,0.000000
92597,zskLB4tU,2024-05-22,0.122,0.228,NaN,NaN,0.000000,0.00,False,False,True,NaN,None,19.990909,NaN,0.000000,0.228,0.476214,True,0.000000


In [8]:
from pathlib import Path
Path("cache/step4/").mkdir(parents=True, exist_ok=True)

df_list = [
    (df3, "df_stats_per_day_per_participant"),
    (df_estimates, "df_hot_water_per_participant") 
]

for df_tuple in df_list: 
    df = df_tuple[0]
    name = df_tuple[1] 
    df.to_parquet(f"cache/step4/{name}.parquet", engine="pyarrow")
    print(f"Created .parquet file for {name}, length: {len(df)}") 

# Add a text file which notes when we last ran this code:
filepath = "cache/step4/notebook_4_last_run_date.txt"

with open(filepath, 'w') as file:
    file.write(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

Created .parquet file for df_stats_per_day_per_participant, length: 92599
Created .parquet file for df_hot_water_per_participant, length: 174
